# Create Customer Order Summary
1. Join the tables Silver_Customers, Silver_addresses and Silver_Orders
2. Retrieve the latest Address of the Customer
3. Calculate the following values
    - total_orders
    - total_items_ordered
    - total_order_amount

![image_1779280970391.png](./imagens/image_1779280970391.png "image_1779280970391.png")

## CircuitBox Data Model
![image_1779281513811.png](./imagens/image_1779281513811.png "image_1779281513811.png")

## Customer Order Summary Requirement
![image_1779281706041.png](./imagens/image_1779281706041.png "image_1779281706041.png")

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW gold_customer_order_summary
AS
SELECT c.customer_id,
       c.customer_name,
       c.date_of_birth,
       c.telephone,
       c.email,
       c.address_line_1,
       c.city,
       c.state,
       c.postcode,
       COUNT(DISTINCT o.order_id) AS total_orders,
       SUM(o.item_quantity) AS total_items_ordered,
       SUM(o.item_quantity * o.item_price) AS total_order_amount
FROM silver_customer c
JOIN silver_addresses a ON c.customer_id = a.customer_id
JOIN silver_orders o ON c.customer_id = o.customer_id
WHERE a.__END_AT IS NULL
GROUP BY ALL;

### Estudo Multiple Catalogs
`https://www.databricks.com/blog/publish-multiple-catalogs-and-schemas-single-dlt-pipeline`

### Codigo DLT

In [0]:
from pyspark.sql import functions as F
from pyspark import pipelines as dp

@dp.table(
    name="bronze_customers",
    comment="Raw orders data ingested from operational_data volume",
    table_properties={"quality": "bronze"}
)
def bronze_customers():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.partitionColumns", "")
        .load("/Volumes/circuitbox/landing/operational_data/customers")
        .select(
            "*",
            "_metadata.file_path",
            F.current_timestamp().alias("ingestion_timestamp")
        )
    )

@dp.table(
    name = "bronze_addresses",
    table_properties = {'quality' : 'bronze'},
    comment = "Raw addresses data ingested from the source system"
)
def bronze_addresses():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.partitionColumns", "")
        .load("/Volumes/circuitbox/landing/operational_data/addresses")
        .select(
            "*",
            "_metadata.file_path",
            F.current_timestamp().alias("ingestion_timestamp")
        )
    )

@dp.table(
    name="bronze_orders",
    table_properties={"quality": "bronze"},
    comment="Raw orders data ingested from operational_data volume"
)
def bronze_orders():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.partitionColumns", "")
        .load("/Volumes/circuitbox/landing/operational_data/orders")
        .select(
            "*",
            "_metadata.file_path",
            F.current_timestamp().alias("ingestion_timestamp")
        )
    )

@dp.table(
    name="silver_customers_clean",
    comment="Cleaned customers data",
    table_properties={"quality": "silver"}
)
@dp.expect_or_fail("valid_customer_id", "customer_id IS NOT NULL")
@dp.expect_or_drop("valid_customer_name", "customer_name IS NOT NULL")
@dp.expect("valid_telephone", "LENGTH(telephone) >= 10")
@dp.expect("valid_email", "email IS NOT NULL")
@dp.expect("valid_date_of_birth", "date_of_birth >= '1920-01-01'")
def silver_customers_clean():
    return (
        spark.readStream
        .table("bronze_customers")
        .select(
            "customer_id",
            "customer_name",
            F.col("date_of_birth").cast("date").alias("date_of_birth"),
            "telephone",
            "email",
            F.col("created_date").cast("date").alias("created_date"),
            F.col("ingestion_timestamp").alias("updated_at")
        )
    )

@dp.table(
    name="silver_addresses_clean",
    comment="Cleaned addresses data",
    table_properties={"quality" : "silver"}
)
@dp.expect_or_fail("valid_customer_id", "customer_id IS NOT NULL")
@dp.expect_or_drop("valid_address", "address_line_1 IS NOT NULL")
@dp.expect("valid_postcode", "LENGTH(postcode) = 5")
def silver_addresses_clean():
    return (
        spark.readStream
        .table("bronze_addresses")
        .select(
            "customer_id",
            "address_line_1",
            "city",
            "state",
            "postcode",
            F.col("created_date").cast("date"),
            F.col("ingestion_timestamp").alias("updated_at")
        )
    )

@dp.table(
    name="silver_orders_clean",
    comment="Cleaned orders data",
    table_properties={"quality": "silver"}
)
@dp.expect_or_fail("valid_customer_id", "customer_id IS NOT NULL")
@dp.expect_or_fail("valid_order_id", "order_id IS NOT NULL")
@dp.expect("valid_order_status", "order_status IN ('Pending', 'Shipped', 'Cancelled', 'Completed')")
@dp.expect("valid_payment_method", "payment_method IN ('Credit Card', 'Bank Transfer', 'PayPal')")
def silver_orders_clean():
    return (
        spark.readStream.table("bronze_orders")
        .select(
            "order_id",
            "customer_id",
            F.col("order_timestamp").cast("timestamp").alias("order_timestamp"),
            "payment_method",
            "items",
            "order_status"
        )
    )


dp.create_streaming_table(
    name="silver_customers",
    comment="SCD Type 1 customers data - maintains latest state per customer",
    table_properties={"quality": "silver"}
)

dp.create_auto_cdc_flow(
    target="silver_customers",
    source="silver_customers_clean",
    keys=["customer_id"],
    sequence_by="updated_at",
    stored_as_scd_type=1,
    ignore_null_updates=True
)

dp.create_streaming_table(
    name="silver_addresses",
    comment="SCD Type 2 addresses data - maintains latest state per addresses",
    table_properties={"quality": "silver"}
)
dp.create_auto_cdc_flow(
    target="silver_addresses",
    source="silver_addresses_clean",
     keys=["customer_id"],
    sequence_by="updated_at",
    stored_as_scd_type=2,
    ignore_null_updates=True
)

@dp.view
def silver_orders_source():
    return (
        spark.readStream.table("silver_orders_clean")
        .select(
            "order_id",
            "customer_id",
            "order_timestamp",
            "payment_method",
            "order_status",
            F.explode("items").alias("item")
        )
        .select(
            "order_id",
            "customer_id",
            "order_timestamp",
            "payment_method",
            "order_status",
            F.col("item.item_id").alias("item_id"),
            F.col("item.name").alias("item_name"),
            F.col("item.price").alias("item_price"),
            F.col("item.quantity").alias("item_quantity"),
            F.col("item.category").alias("item_category")
        )
    )

dp.create_streaming_table(
    name="silver_orders",
    comment="SCD Type 1 orders data - maintains latest state per order item",
    table_properties={"quality": "silver"}
)

dp.create_auto_cdc_flow(
    target="silver_orders",
    source="silver_orders_source",
    keys=["order_id", "item_id"],
    sequence_by="order_timestamp",
    stored_as_scd_type=1,
    ignore_null_updates=True
)

@dp.materialized_view(
    name="gold_customer_order_summary",
    comment="Aggregated customer order summary for analytics and reporting",
    table_properties={"quality": "gold"}
)
@dp.expect_or_fail("valid_customer_id", "customer_id IS NOT NULL")
@dp.expect("valid_total_orders", "total_orders > 0")
@dp.expect("valid_total_amount", "total_order_amount >= 0")
def gold_customer_order_summary():
    " Creates gold layer table with customer order summary aggregations "
    df_customers = spark.read.table("silver_customers")
    df_addresses = spark.read.table("silver_addresses")
    df_orders = spark.read.table("silver_orders")
    
    df_addresses_active = df_addresses.filter(F.col("__END_AT").isNull())
    
    df_joined = (
        df_customers
        .join(df_addresses_active, "customer_id", "inner")
        .join(df_orders, "customer_id", "inner")
    )
    
    df_result = (
        df_joined
        .groupBy(
            "customer_id",
            "customer_name",
            "date_of_birth",
            "telephone",
            "email",
            "address_line_1",
            "city",
            "state",
            "postcode"
        )
        .agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.sum("item_quantity").alias("total_items_ordered"),
            F.sum(F.col("item_quantity") * F.col("item_price")).alias("total_order_amount")
        )
    )
    
    return df_result
